# World Cup Agent Arena — Build-Day Walkthrough

Welcome! By the end of this notebook you'll have run a complete **trading agent** from start to finish. Use it as the template for the agent you'll build to compete in the arena.

## What is this, in plain terms?

- **The arena** is a competition. Your agent looks at upcoming World Cup 2026 matches, predicts who will win, and (optionally) places play-money bets on those outcomes. You're scored on how good your predictions and trades are.
- **An agent** is just a program that runs a loop: **gather data → think → act → record why it acted.** This notebook walks through one full pass of that loop for a single match.

## The flow (run the cells top to bottom)

| Step | What it does |
|------|--------------|
| Setup | Set your keys and shared settings |
| 1 | **Find the matches** and pick one to analyze |
| 2 | **Fetch the match's pre-game data** (model predictions, odds, expected goals) and summarize it |
| 3 | **Fetch the betting market** and its live prices, and summarize it |
| 4 | **Fetch historical team stats** and summarize them |
| 5 | **Predict the result** — the agent forms its own opinion, ignoring the market |
| 6 | **Decide whether to bet** — compare the agent's opinion to the market |
| 7 | **Place the bet** (open a position) — or skip if there's no edge |
| 8 | **Record the agent's reasoning** to the ledger so the arena can audit and score it |

## Glossary (skim this first)

| Term | What it means |
|------|---------------|
| **Fixture** | A single scheduled match (e.g. Mexico vs South Africa). "Fixture" is just the sports-data word for "game." |
| **Sportmonks** | A sports-data provider. We use it for schedules, team info, model predictions, and odds. |
| **Bookmaker odds** | The prices a betting company offers on each outcome. They imply a probability (e.g. odds that imply "home wins 55% of the time"). |
| **Expected goals (xG)** | A stat estimating how many goals a team *should* have scored based on the quality of their chances. |
| **Polymarket** | A prediction market where people trade on real-world outcomes. Prices move like a stock and reflect the crowd's implied probability. |
| **Moneyline** | The "who wins?" market: three outcomes — home win, draw, away win. |
| **Event slug** | Polymarket's human-readable id for a market, e.g. `fifwc-mex-rsa-2026-06-11`. We use it to look the market up. |
| **Mid price** | The midpoint between the best buy and sell price for an outcome, from 0 to 1. A mid of 0.62 ≈ a 62% implied chance. |
| **Edge** | (your probability − the market's probability). Positive edge = the market is underpricing your pick → maybe worth a bet. |
| **Supabase** | The database holding the arena's extra historical stats. |
| **Ledger** | A structured log of every step the agent took and why. The arena reads it to verify and score your agent. |
| **LLM digest** | We ask Claude to boil a big, noisy API response down to a small, clean JSON summary so later steps stay simple. |

**Before running:** in the **Setup** cell, replace the two placeholder credentials — `ARENA_KEY` (mint at https://staging.stair-ai.com/api-keys) and `ANTHROPIC_KEY` (get one at https://console.anthropic.com). The Supabase URL and key are shared across all builders and already filled in.

## Setup — keys, endpoints, and shared settings

This cell defines everything the rest of the notebook reuses: your two API keys, the arena's proxy URLs, and a few constants. **You only need to edit the two placeholder keys** — everything else already works on staging.

| Setting | What it is |
|---------|------------|
| `ARENA_KEY` | **← you set this.** Your arena API key; authenticates every arena call. |
| `ANTHROPIC_KEY` | **← you set this.** Your Anthropic key, used for the Claude "digest" / reasoning calls. |
| `ARENA` | Base URL of the arena (staging). |
| `SPORTMONKS_PROXY` / `POLYMARKET_*` | Arena **proxy** URLs. You call the arena; it forwards to Sportmonks / Polymarket with its own upstream keys, so you never need theirs. |
| `SUPABASE` / `SUPABASE_KEY` | Shared, read-only database access. Already filled in. |
| `SPORTMONKS_SEASON_ID` | The World Cup 2026 season id — the only tournament this guide uses. |
| `LLM_MODEL`, `LLM_*` | Which Claude model to use and its limits (including "extended thinking," where the model exposes its reasoning). |

`_extract()` is a small helper that pulls the final answer text and the model's thinking out of a Claude response. You'll see it used after every LLM call.

In [ ]:
import os, json, time, uuid, requests

ARENA            = "https://staging.stair-ai.com"
SPORTMONKS_PROXY = f"{ARENA}/api/v1/data/proxy/sportmonks/v3/football"
POLYMARKET_CLOB  = f"{ARENA}/api/v1/data/proxy/polymarket-clob"
POLYMARKET_GAMMA = f"{ARENA}/api/v1/data/proxy/polymarket-gamma"
ARENA_KEY        = "FILL IN YOUR ARENA KEY HERE"
# Staging shares a single publishable Supabase key for every builder — no
# per-account JWT, no extra setup. The arena will publish these two values
# alongside the API key minted in the portal.
SUPABASE         = "https://ezvbmtvrvzageqixvdak.supabase.co"
SUPABASE_KEY     = "sb_publishable__m8bOkD05ToFwATpaWST5w_2-3fGS7V"
ANTHROPIC_KEY    = "FILL IN YOUR ANTHROPIC KEY HERE"

# --- Other LLM providers (OPTIONAL) ------------------------------------------
# This notebook calls Anthropic by default. To use a DIFFERENT provider instead:
#   1. paste its key below,
#   2. pip install its SDK (see the optional section in requirements.txt),
#   3. uncomment its client in the next code cell, and
#   4. in each LLM cell, comment out the Anthropic block and UNCOMMENT the block
#      for your provider.
# The _extract() / _mi() helpers already understand all four response shapes, so
# nothing else has to change.
GEMINI_API_KEY   = "FILL IN YOUR GOOGLE GEMINI KEY HERE"    # Google AI Studio: https://aistudio.google.com/apikey
OPENAI_API_KEY   = "FILL IN YOUR OPENAI KEY HERE"           # OpenAI:           https://platform.openai.com/api-keys
DEEPSEEK_API_KEY = "FILL IN YOUR DEEPSEEK KEY HERE"         # DeepSeek:         https://platform.deepseek.com/api_keys
H_ARENA          = {"x-api-key": ARENA_KEY}
H_WCA            = {"apikey": SUPABASE_KEY, "Accept-Profile": "world_cup_arena"}

# Tournament constant — WC2026 is the only season this guide targets.
SPORTMONKS_SEASON_ID = 26618

# Reasoning-Ledger schema constants (per schema/records.schema.json v0.3 in
# StairAI/Reasoning-Ledger). agent_id is NOT set client-side: the arena
# resolves it server-side from the x-api-key on POST, so the wire records
# omit it. The local dump produced by this script also omits it for fidelity
# with what the agent actually transmits.
LEDGER_SCHEMA_VERSION = "0.3"
# The model each provider should use. LLM_MODEL stays the Anthropic model so the
# default path is unchanged; the others are only used if you switch providers.
LLM_MODEL             = "claude-haiku-4-5-20251001"   # Anthropic (default)
GEMINI_MODEL          = "gemini-2.0-flash"            # Google Gemini
OPENAI_MODEL          = "gpt-4o-mini"                 # OpenAI
DEEPSEEK_MODEL        = "deepseek-chat"               # DeepSeek (use "deepseek-reasoner" for a thinking trace)

# Anthropic extended-thinking knobs. budget_tokens must be < max_tokens; when
# enabled, response.content contains both `thinking` and `text` blocks — see
# scripts/model_reasoning_blocks.ipynb (Pattern A) for the canonical reference.
LLM_MAX_TOKENS      = 2400
LLM_THINKING_BUDGET = 1024
LLM_THINKING        = {"type": "enabled", "budget_tokens": LLM_THINKING_BUDGET}


def _extract(resp):
    """Return (final_text, thinking_text) from ANY of the four providers, so the
    rest of the notebook stays provider-agnostic:
      - Anthropic        : resp.content is a list of typed blocks (text/thinking)
      - OpenAI & DeepSeek: resp.choices[0].message.content (+ reasoning_content,
                           which DeepSeek's 'deepseek-reasoner' model returns)
      - Gemini           : resp.text (Gemini hides its thinking by default)"""
    # Anthropic
    if hasattr(resp, "content") and isinstance(resp.content, list):
        text_parts, thinking_parts = [], []
        for block in resp.content:
            if block.type == "thinking":
                thinking_parts.append(block.thinking)
            elif block.type == "text":
                text_parts.append(block.text)
        return "".join(text_parts), "\n\n".join(thinking_parts)
    # OpenAI / DeepSeek (OpenAI-compatible)
    if hasattr(resp, "choices"):
        msg = resp.choices[0].message
        return (msg.content or ""), (getattr(msg, "reasoning_content", "") or "")
    # Gemini
    if hasattr(resp, "text"):
        return (resp.text or ""), ""
    raise TypeError(f"Unrecognized LLM response type: {type(resp)!r}")


# --- Sanity check: catch a placeholder key NOW, not 6 cells from now. ---
_missing = [n for n, v in [("ARENA_KEY", ARENA_KEY), ("ANTHROPIC_KEY", ANTHROPIC_KEY)]
            if "FILL IN" in v]
if _missing:
    print(f"WARNING: still need to set {', '.join(_missing)} (edit this cell first).")
else:
    print("Both API keys are set.")
print(f"Arena  : {ARENA}")
print(f"Model  : {LLM_MODEL}")
print(f"Season : World Cup 2026 (id {SPORTMONKS_SEASON_ID})")
print("Setup complete -- run the cells below in order.")


## Step 1 · Find the matches (fixtures) and pick one

A **fixture** is one scheduled match. Before the agent can analyze anything, it needs to know which matches exist — so we ask Sportmonks for the **season schedule**: every stage, round, and fixture for World Cup 2026.

We call the arena's Sportmonks **proxy** (not Sportmonks directly): the arena forwards the request with its own Sportmonks key and wraps the reply in an envelope — `{body, statusCode, requestId, …}` — so we "peel" `body` → `data` to get the actual schedule.

For this walkthrough we hard-code the opener, **Mexico vs South Africa** (`fixture_id 19609127`), as the fixture to reason about. Your agent would instead loop over the schedule and pick fixtures itself.

- Original Sportmonks endpoint: `GET /v3/football/schedules/seasons/{SEASON_ID}`
- Sportmonks WC2026 guide: https://docs.sportmonks.com/v3/world-cup-2026/how-to-build-your-world-cup-application

In [ ]:
r = requests.get(
    f"{SPORTMONKS_PROXY}/schedules/seasons/{SPORTMONKS_SEASON_ID}",
    headers=H_ARENA, timeout=10,
)
r.raise_for_status()

# Every arena proxy call wraps the upstream reply in an "envelope":
#   {body, duration, statusCode, requestId, _proxy, headers}
# The real Sportmonks payload lives under envelope["body"]["data"].
envelope = r.json()
schedule = envelope["body"]["data"]

print(f"HTTP {r.status_code} (OK) -- the arena answered.")
print(f"Envelope keys from the proxy: {list(envelope.keys())}")
print(f"Found {len(schedule)} schedule entries (stages / rounds / fixtures) for WC2026.\n")

# A real agent would scan `schedule` and pick fixtures itself. For this guide we
# hard-code the tournament opener so everyone analyzes the same match:
#   Mexico (MEX) vs South Africa (ZAF) -- 2026-06-11 -- fixture_id 19609127
SPORTMONKS_FIXTURE_ID = 19609127
print(f"Chosen fixture: Mexico vs South Africa (fixture_id {SPORTMONKS_FIXTURE_ID})")


### Find the matching Polymarket market (the "event slug")

To bet on a match we need its market on Polymarket. The arena keeps a curated **fixture ↔ Polymarket-event mapping** so you don't have to match them by hand.

We only need one field from it — the **`polymarket_event_slug`**, Polymarket's id for this match's market (e.g. `fifwc-mex-rsa-2026-06-11`). Everything else about the market (prices, token ids) we fetch live from Polymarket in Step 3. If a fixture has no mapping, the slug is `None` and the agent simply runs in predict-only mode.

In [ ]:
r = requests.get(
    f"{ARENA}/api/v1/web/mapping",
    params={"fixture_id": SPORTMONKS_FIXTURE_ID},
    headers=H_ARENA, timeout=10,
)
r.raise_for_status()
mappings = r.json().get("mappings") or []
polymarket_event_slug = mappings[0]["polymarket_event_slug"] if mappings else None
print(f"HTTP {r.status_code} (OK)")
if polymarket_event_slug:
    print(f"This fixture maps to Polymarket event slug: {polymarket_event_slug!r}")
    print("We'll use this slug in Step 3 to pull the live market.")
else:
    print("No Polymarket market is mapped to this fixture.")
    print("That's fine -- the agent will run in predict-only mode (no betting).")


## Step 2 · Fetch the match's pre-game data and summarize it

Now pull the **pre-game signals** for our chosen fixture. We ask Sportmonks to "include" several related pieces of data in one call. (The `statistics` include only fills in *after* a match, so we skip it and request these instead:)

| Include | What it gives us |
|---------|------------------|
| `participants` | The two teams, with home/away labels and short codes (e.g. MEX, RSA) |
| `predictions` | Sportmonks' own machine-learning model probabilities for win / draw / loss |
| `odds` | Bookmaker prices — we'll average them into a "consensus" probability |
| `xGFixture` | Expected-goals projection for each team |

Same envelope as Step 1, so again we peel `body` → `data`. We then split the two `participants` into `home` and `away`.

- Sportmonks doc: https://docs.sportmonks.com/v3/endpoints-and-entities/endpoints/fixtures/get-fixture-by-id

In [ ]:
r = requests.get(
    f"{SPORTMONKS_PROXY}/fixtures/{SPORTMONKS_FIXTURE_ID}",
    params={"include": "participants;predictions;odds;xGFixture"},
    headers=H_ARENA, timeout=60,
)
r.raise_for_status()
fixture = r.json()["body"]["data"]    # same envelope as Step 1: peel body -> data

home = next(p for p in fixture["participants"] if p["meta"]["location"] == "home")
away = next(p for p in fixture["participants"] if p["meta"]["location"] == "away")

print(f"HTTP {r.status_code} (OK)")
print(f"Fixture : {fixture['name']}")
print(f"Kickoff : {fixture.get('starting_at')}")
print(f"Home    : {home['name']} ({home['short_code']})")
print(f"Away    : {away['name']} ({away['short_code']})")
print("\nHow much pre-game data came back? (empty rows are possible on staging)")
print(f"  - Sportmonks model predictions : {len(fixture.get('predictions') or [])} rows")
print(f"  - bookmaker odds               : {len(fixture.get('odds') or [])} rows")
print(f"  - expected-goals (xG)          : {len(fixture.get('xgfixture') or [])} rows")


### Summarize the match data with an LLM

The raw Sportmonks payload is large and noisy. Here we hand it to Claude with strict instructions to return a **small, clean JSON summary** (a "digest"): model probabilities, bookmaker consensus, expected goals, plus an honest note of what's missing.

Why bother? Later steps (the prediction in Step 5) only need the distilled signals, not hundreds of raw rows. Digesting now keeps every downstream prompt small, cheap, and consistent. This "fetch → digest" pattern repeats in Steps 3 and 4.

Note: we enable **extended thinking**, so the response has both a `thinking` trace and the final `text`. `_extract()` separates them, and a regex pulls the JSON object out of the text.

In [ ]:
import anthropic
client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

# To use a different provider, uncomment its client here (after pip-installing the
# SDK), then uncomment that provider's call block in each LLM cell below.
# from google import genai                                    # pip install google-genai
# gemini_client   = genai.Client(api_key=GEMINI_API_KEY)
# from openai import OpenAI                                   # pip install openai
# openai_client   = OpenAI(api_key=OPENAI_API_KEY)
# deepseek_client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")

DIGEST_SYS = (
    "You are a soccer analyst. You receive a raw Sportmonks pre-match payload for "
    "one fixture and must distil it into a self-contained JSON digest that a "
    "downstream LLM (with no other context about Sportmonks) will read.\n\n"

    "## Input shape\n"
    "  - fixture       : match name (e.g. 'Mexico vs South Africa')\n"
    "  - home_code     : home team short code (use as a JSON key for the home outcome)\n"
    "  - away_code     : away team short code (use as a JSON key for the away outcome)\n"
    "  - predictions[] : Sportmonks ML model rows. Each row has `type_id` (numeric — "
    "                    the Full-Time-Result / 1X2 winner type carries win/draw/loss "
    "                    probabilities) and a `predictions` object with the numeric "
    "                    probability values. May be empty if Sportmonks has no model "
    "                    output for this fixture.\n"
    "  - odds[]        : bookmaker odds rows. Each row is ONE bookmaker's price for "
    "                    ONE outcome of ONE market. Key fields: `bookmaker_id`, "
    "                    `market_id` (1 = Full-Time-Result / 1X2 winner — IGNORE other "
    "                    markets), `label` ('1' home, 'X' draw, '2' away), `value` "
    "                    (decimal odds), `probability` (implied probability as a "
    "                    percentage 0-100). For the consensus, average `probability` "
    "                    across all bookmakers for market_id == 1 only, then divide by "
    "                    100 to express as 0..1. May be empty.\n"
    "  - xGFixture[]   : expected-goals entries per team. Each row has "
    "                    `participant_id` (team id matching home/away participant) and "
    "                    `value` (xG number). May be empty.\n\n"

    "## Output schema (return ONLY this JSON — no prose, no code fences)\n"
    "{\n"
    "  'fixture'                       : str,                                                          // echo input\n"
    "  'home_team'                     : str,                                                          // home_code\n"
    "  'away_team'                     : str,                                                          // away_code\n"
    "  'sportmonks_ml_win_prob'        : {home_code: float, 'draw': float, away_code: float} | null,   // probabilities in 0..1; sum ≈ 1\n"
    "  'bookmaker_consensus_win_prob'  : {home_code: float, 'draw': float, away_code: float} | null,   // probabilities in 0..1\n"
    "  'bookmaker_count'               : int | null,                                                   // bookmakers averaged into consensus\n"
    "  'expected_goals'                : {home_code: float, away_code: float} | null,                  // xG per side\n"
    "  'data_availability': {                                                                          // honest reporting so downstream knows what's missing\n"
    "    'sportmonks_ml'        : 'available' | 'missing',\n"
    "    'bookmaker_consensus'  : 'available' | 'missing',\n"
    "    'expected_goals'       : 'available' | 'missing'\n"
    "  },\n"
    "  'summary': str   // 1-3 sentences. MUST be readable in isolation by an LLM that has no other Sportmonks context. Name the available signals and what they imply; if everything is missing, say so plainly. Mention the home/away team codes by name.\n"
    "}\n\n"

    "Use null (not 0) when source data is missing. Do NOT fabricate values."
)

# The user message is identical across providers, so build it once.
digest_input = json.dumps({
    "fixture":     fixture["name"],
    "home_code":   home["short_code"],
    "away_code":   away["short_code"],
    "predictions": fixture.get("predictions"),
    "odds":        sorted(
        [o for o in (fixture.get("odds") or []) if o.get("market_id") == 1],
        key=lambda o: o.get("latest_bookmaker_update") or "",
    )[-1:],  # latest single 1X2 odds row (full odds list overflows the context window)
    "xGFixture":   fixture.get("xgfixture"),    # field is lowercase despite include name
})

# === Anthropic (default) =====================================================
llm_digest = client.messages.create(
    model=LLM_MODEL,
    max_tokens=LLM_MAX_TOKENS,
    thinking=LLM_THINKING,
    system=DIGEST_SYS,
    messages=[{"role": "user", "content": digest_input}],
)

# === Gemini -- uncomment to use (and comment out the Anthropic block above) ===
# llm_digest = gemini_client.models.generate_content(
#     model=GEMINI_MODEL,
#     contents=digest_input,
#     config={"system_instruction": DIGEST_SYS, "max_output_tokens": LLM_MAX_TOKENS},
# )

# === OpenAI -- uncomment to use ==============================================
# llm_digest = openai_client.chat.completions.create(
#     model=OPENAI_MODEL,
#     max_tokens=LLM_MAX_TOKENS,
#     messages=[{"role": "system", "content": DIGEST_SYS},
#               {"role": "user",   "content": digest_input}],
# )

# === DeepSeek -- uncomment to use ============================================
# llm_digest = deepseek_client.chat.completions.create(
#     model=DEEPSEEK_MODEL,
#     max_tokens=LLM_MAX_TOKENS,
#     messages=[{"role": "system", "content": DIGEST_SYS},
#               {"role": "user",   "content": digest_input}],
# )

raw, thinking_digest = _extract(llm_digest)

# Claude returns the digest as text; pull the {...} object out of it
# (re.DOTALL lets the regex span newlines; also strips any prose/code fences).
import re
match = re.search(r"\{.*\}", raw, re.DOTALL)
sportmonks_digest = json.loads(match.group(0)) if match else None

print(f"Claude reasoned for {len(thinking_digest)} chars before answering.")
print("Clean digest the rest of the notebook will use:\n")
print(json.dumps(sportmonks_digest, indent=2))


## Step 3 · Fetch the betting market and its prices, then summarize it

Now we get the actual **market** we could trade on. The "who wins?" market is the **moneyline**, with three outcomes: home win, draw, away win. On Polymarket each outcome is its own yes/no market, and the three are grouped under one **event**.

Polymarket exposes two APIs (both reached through the arena proxy):

| API | What we ask it | What we get back |
|-----|----------------|------------------|
| **Gamma** (`/events?slug=…`) | the event by its slug | the event + its 3 child markets, each with a `conditionId` and `clobTokenIds` (a YES and a NO token) |
| **CLOB** (`/midpoint?token_id=…`) | a YES token's price | the live **mid** price (0–1), i.e. the implied probability of that outcome |

So the recipe is: call Gamma once to get the three markets and their token ids → call CLOB once per YES token for its live price → assemble one tidy `moneyline` dict.

How we label each market: Polymarket's WC2026 events follow a naming convention — the event "ticker" is `fifwc-{home}-{away}-{YYYY-MM-DD}`, and each child market's slug is that ticker plus `-{team_code}` or `-draw`. We parse those to tag each market as home / draw / away.

In [ ]:
import re
TICKER_RE = re.compile(r"^fifwc-([a-z]{2,4})-([a-z]{2,4})-(\d{4}-\d{2}-\d{2})$")


def _clob_mid(token_id_str: str | None) -> float | None:
    """Single CLOB midpoint call. Polymarket's CLOB takes the token id as a
    decimal string (the raw value is a 78-digit integer)."""
    if not token_id_str:
        return None
    try:
        resp = requests.get(
            f"{POLYMARKET_CLOB}/midpoint",
            params={"token_id": token_id_str},
            headers=H_ARENA, timeout=10,
        )
        if not resp.ok:
            return None
        body = resp.json().get("body")
        if isinstance(body, dict) and "mid" in body:
            return float(body["mid"])
    except Exception:
        pass
    return None


def _outcome_from_market_slug(market_slug: str, ticker: str,
                              home_code: str, away_code: str) -> str | None:
    """Map a child-market slug ('fifwc-mex-rsa-2026-06-11-mex') to an
    outcome key ('home' | 'draw' | 'away')."""
    if not market_slug.startswith(ticker + "-"):
        return None
    suffix = market_slug[len(ticker) + 1:]
    if suffix == home_code: return "home"
    if suffix == "draw":    return "draw"
    if suffix == away_code: return "away"
    return None


if not polymarket_event_slug:
    moneyline = None
else:
    # 3a · Gamma: one call returns the event + its 3 child markets.
    r = requests.get(
        f"{POLYMARKET_GAMMA}/events",
        params={"slug": polymarket_event_slug},
        headers=H_ARENA, timeout=15,
    )
    r.raise_for_status()
    events = r.json().get("body") or []
    event  = events[0] if events else None

    if event is None:
        moneyline = None
    else:
        ticker = (event.get("ticker") or "").lower()
        m = TICKER_RE.match(ticker)
        if not m:
            moneyline = None
        else:
            pm_home_code, pm_away_code, _ = m.groups()
            outcomes: dict[str, dict] = {}
            for mkt in (event.get("markets") or []):
                key = _outcome_from_market_slug((mkt.get("slug") or "").lower(),
                                                ticker, pm_home_code, pm_away_code)
                if key is None:
                    continue
                # clobTokenIds is a JSON-encoded string: [YES_token, NO_token].
                try:
                    token_ids = json.loads(mkt.get("clobTokenIds") or "[]")
                except json.JSONDecodeError:
                    token_ids = []
                token_yes = token_ids[0] if token_ids else None
                outcomes[key] = {
                    "team_code":       key if key == "draw" else (
                                            pm_home_code.upper() if key == "home"
                                            else pm_away_code.upper()),
                    "condition_id":    mkt.get("conditionId"),
                    "token_yes":       token_yes,
                    "current_mid_yes": _clob_mid(token_yes),  # 3b · one CLOB call per YES token
                }

            moneyline = {
                "sportmonks_match_id":   SPORTMONKS_FIXTURE_ID,
                "fixture":               event.get("title"),
                "kickoff_utc":           event.get("startDate"),
                "polymarket_event_slug": polymarket_event_slug,
                "outcomes":              outcomes,
            }

if moneyline is None:
    print("No tradable Polymarket market for this fixture -- predict-only mode.")
else:
    n_mids = sum(1 for o in moneyline["outcomes"].values()
                 if o["current_mid_yes"] is not None)
    print(f"Built the 3-way moneyline for: {moneyline['fixture']}")
    print(f"Live mid prices retrieved for {n_mids}/3 outcomes (home / draw / away).")
    print("Full market (prices + the token ids needed to place an order):\n")
print(json.dumps(moneyline, indent=2, default=str))


### Summarize the market with an LLM

Same pattern as Step 2: distill the raw market response into a self-contained JSON. The digest reports each outcome's **implied probability** (from the mid prices), whether the probabilities sum to ~1 (a sanity check for stale prices), and the **execution handles** (`condition_id` + `token_yes`) a later step needs to actually place an order. If there's no market for this fixture, we emit a clearly-labeled "no market" digest instead.

In [ ]:
POLYMARKET_DIGEST_SYS = (
    "You are an analyst digesting a Polymarket moneyline (3-way match-winner) "
    "market response into a self-contained JSON for a downstream LLM that has "
    "no other Polymarket context.\n\n"

    "## Input shape\n"
    "  - sportmonks_match_id   : numeric fixture id (echo)\n"
    "  - fixture               : match name (e.g. 'Mexico vs South Africa')\n"
    "  - kickoff_utc           : ISO kickoff timestamp\n"
    "  - polymarket_event_slug : Polymarket event slug grouping the 3 binary markets\n"
    "  - outcomes.{home,draw,away}\n"
    "      team_code           : team short code (or 'draw' for the draw outcome)\n"
    "      condition_id        : Polymarket condition id (needed for trade execution)\n"
    "      token_yes           : ERC1155 YES-side token id (buy YES to back the outcome)\n"
    "      current_mid_yes     : midpoint price of the YES token in 0..1 == implied probability\n"
    "                            of that outcome winning. null if CLOB lookup failed.\n\n"

    "## Output schema (return ONLY this JSON — no prose, no code fences)\n"
    "{\n"
    "  'fixture'              : str,\n"
    "  'market_handle'        : str,                                                          // polymarket_event_slug\n"
    "  'implied_win_prob'     : {home_code: float, 'draw': float, away_code: float} | null,   // from current_mid_yes; null if unavailable\n"
    "  'sum_implied_prob'     : float | null,                                                 // should be ≈1.0; outside [0.95, 1.10] = stale prices or arb gap\n"
    "  'execution_handles'    : {home_code: {condition_id, token_yes},                        // for the downstream trade-execution step\n"
    "                            'draw'   : {condition_id, token_yes},\n"
    "                            away_code: {condition_id, token_yes}},\n"
    "  'data_availability'    : 'mids_available' | 'mids_partial' | 'mids_missing' | 'no_market',\n"
    "  'summary'              : str   // 1-3 sentences self-contained. Name the favorite (highest implied prob), the spread, and any anomaly. If mids are missing, say so plainly and identify what's still available (execution handles can still be used to place orders blind).\n"
    "}\n\n"

    "Use null when input shows null. Do NOT fabricate prices."
)

if moneyline is None:
    polymarket_digest = {
        "fixture":              None,
        "market_handle":        None,
        "implied_win_prob":     None,
        "sum_implied_prob":     None,
        "execution_handles":    None,
        "data_availability":    "no_market",
        "summary":              f"No Polymarket moneyline mapping for Sportmonks fixture "
                                f"{SPORTMONKS_FIXTURE_ID}. The fixture either isn't listed on "
                                f"Polymarket yet or its curated mapping is marked no_match.",
    }
else:
    pm_input = json.dumps(moneyline)

    # === Anthropic (default) =================================================
    llm_pm = client.messages.create(
        model=LLM_MODEL,
        max_tokens=LLM_MAX_TOKENS,
        thinking=LLM_THINKING,
        system=POLYMARKET_DIGEST_SYS,
        messages=[{"role": "user", "content": pm_input}],
    )

    # === Gemini -- uncomment to use (comment out the Anthropic block above) ==
    # llm_pm = gemini_client.models.generate_content(
    #     model=GEMINI_MODEL,
    #     contents=pm_input,
    #     config={"system_instruction": POLYMARKET_DIGEST_SYS, "max_output_tokens": LLM_MAX_TOKENS},
    # )

    # === OpenAI -- uncomment to use =========================================
    # llm_pm = openai_client.chat.completions.create(
    #     model=OPENAI_MODEL,
    #     max_tokens=LLM_MAX_TOKENS,
    #     messages=[{"role": "system", "content": POLYMARKET_DIGEST_SYS},
    #               {"role": "user",   "content": pm_input}],
    # )

    # === DeepSeek -- uncomment to use =======================================
    # llm_pm = deepseek_client.chat.completions.create(
    #     model=DEEPSEEK_MODEL,
    #     max_tokens=LLM_MAX_TOKENS,
    #     messages=[{"role": "system", "content": POLYMARKET_DIGEST_SYS},
    #               {"role": "user",   "content": pm_input}],
    # )

    raw_pm, thinking_pm = _extract(llm_pm)
    m = re.search(r"\{.*\}", raw_pm, re.DOTALL)
    polymarket_digest = json.loads(m.group(0)) if m else None
    print(f"Claude digested the market ({len(thinking_pm)} chars of thinking).")

print("\nMarket digest (implied probabilities + execution handles):\n")
print(json.dumps(polymarket_digest, indent=2))


## Step 4 · Fetch historical team stats and summarize them

The arena also ships a **database** (Supabase) of deeper historical stats. We use it in three sub-steps:

| Sub-step | What it does |
|----------|--------------|
| **4a · Discover** | Read the catalog to learn which tables exist — no external docs needed |
| **4b · Fetch** | Pull the rows we want for both teams |
| **4c · Digest** | Have Claude summarize them into JSON for Step 5 |

⚠️ **Heads-up — country ids differ between systems.** Sportmonks numbers Mexico = 458 / South Africa = 146, but this database (StatsBomb-derived) uses **147 / 211**. Always use the id that matches the system you're querying.

In [ ]:
TEAM_A_ID = 147   # Mexico
TEAM_B_ID = 211   # South Africa

H_PUBLIC = {"apikey": SUPABASE_KEY}                                       # default schema = public
H_WCA    = {"apikey": SUPABASE_KEY, "Accept-Profile": "world_cup_arena"}  # data layer

print("Querying the Supabase stats DB. Heads-up: country ids differ from Sportmonks.")
print(f"  Mexico       -> country_id {TEAM_A_ID}")
print(f"  South Africa -> country_id {TEAM_B_ID}")


### 4a · Discover what data exists

A good agent that's new to a database **asks what's there first** instead of guessing table names. The arena exposes a self-describing catalog in the `public` schema — one query tells you the tables, their categories, row counts, and descriptions (and, via the columns view, every column's type and meaning). No external docs required.

| View | What it returns |
|------|-----------------|
| `public.catalog_tables` | All available tables and a description of each |
| `public.catalog_columns` | All columns across every table, with types and descriptions |
| `public.catalog_full` | Both combined — tables and their columns in one response |

Below we read `catalog_full`, print every table, then pick **one** priors table (`ads_a_country_style`, which has playing-style indicators) for the example. A real agent could pull several (head-to-head, knockout patterns, etc.) — same pattern, just more rows.

In [ ]:
r = requests.get(
    f"{SUPABASE}/rest/v1/catalog_full",
    params={
        "select": "table_name,category,row_count,table_description",
        "order":  "category,table_name",
    },
    headers=H_PUBLIC, timeout=10,
)
r.raise_for_status()
catalog = r.json()

print(f"HTTP {r.status_code} (OK) -- the catalog lists every table available:\n")
for t in catalog:
    desc = (t.get("table_description") or "-").replace("\n", " ")[:60]
    cat  = t.get("category") or "?"
    print(f"  [{cat:11s}] {t['table_name']:30s}  rows={t['row_count']:>5d}  - {desc}")

# For this example we fetch ONE table, picked from the list above for its
# playing-style indicators. A real agent could pull more (H2H, KO pattern,
# etc.) the same way -- just more rows in the dict.
WANTED_TABLE = "ads_a_country_style"
print(f"\nFor this walkthrough we'll pull stats from: {WANTED_TABLE}")


### 4b · Fetch the stats for both teams

Now query the chosen table for just our two teams. Supabase uses PostgREST-style query params: `country_id=in.(147,211)` means "rows where country_id is 147 or 211," and `select=*` returns all columns. We send the `world_cup_arena` schema header so the request hits the arena's data tables.

In [ ]:
r = requests.get(
    f"{SUPABASE}/rest/v1/{WANTED_TABLE}",
    params={"country_id": f"in.({TEAM_A_ID},{TEAM_B_ID})", "select": "*"},
    headers=H_WCA, timeout=10,
)
r.raise_for_status()
priors_rows = r.json()

print(f"HTTP {r.status_code} (OK) -- pulled '{WANTED_TABLE}' for both teams.")
print(f"Got {len(priors_rows)} row(s) (one per team that has data).")
print("Raw rows (the full stats Claude will summarize next):\n")
print(json.dumps(priors_rows, indent=2, default=str))


### 4c · Summarize the stats with an LLM

The digest pattern once more: Claude turns the raw rows into a compact per-team profile (set-piece efficiency, goals per game, etc.) and flags small-sample caveats. It deliberately does **not** output a win probability — that's Step 5's job.

In [ ]:
SUPABASE_DIGEST_SYS = (
    "You are an analyst aggregating Supabase priors data for one fixture into "
    "a self-contained JSON digest for a downstream LLM that has no other "
    "context about the data layer.\n\n"

    "## Input shape\n"
    "  - fixture        : match name\n"
    "  - source_table   : the Supabase table the rows came from (echo for traceability)\n"
    "  - home_code, away_code : team short codes (use as JSON keys for the output)\n"
    "  - home_id,   away_id   : country ids in this dataset (StatsBomb numbering)\n"
    "  - rows           : list of rows from `ads_a_country_style`, one per country.\n"
    "      Key columns include: country_id, set_piece_shots, set_piece_goals,\n"
    "      conversion_rate, group_matches, group_goals_against, ko_matches,\n"
    "      ko_goals_against, group_gpg (goals/game in group stage),\n"
    "      ko_gpg (goals/game in knockout stage).\n"
    "      Match each row to home_id / away_id by country_id. Sample sizes are\n"
    "      often tiny in this dataset — call that out if it impacts confidence.\n\n"

    "## Output schema (return ONLY this JSON — no prose, no code fences)\n"
    "{\n"
    "  'fixture'      : str,\n"
    "  'source_table' : str,                                              // echo\n"
    "  'teams': {\n"
    "    home_code: {                                                     // team A profile from country_style\n"
    "      'set_piece_efficiency' : float | null,                          // set_piece_goals / set_piece_shots\n"
    "      'set_piece_sample'     : int   | null,                          // set_piece_shots — proxies confidence\n"
    "      'group_goals_per_game' : float | null,\n"
    "      'ko_goals_per_game'    : float | null\n"
    "    },\n"
    "    away_code: { same shape }\n"
    "  },\n"
    "  'data_availability': 'rich' | 'partial' | 'sparse',\n"
    "  'summary': str   // 1-3 sentences self-contained. Name the two teams, the strongest\n"
    "                 // style signal, and call out small-sample caveats. Do NOT give a\n"
    "                 // win probability — that's for §5 combined analysis.\n"
    "}\n\n"

    "Use null when input is empty/missing. Don't fabricate values."
)

sb_input = json.dumps({
    "fixture":      fixture["name"],
    "source_table": WANTED_TABLE,
    "home_code":    home["short_code"],
    "away_code":    away["short_code"],
    "home_id":      TEAM_A_ID,
    "away_id":      TEAM_B_ID,
    "rows":         priors_rows,
}, default=str)

# === Anthropic (default) =====================================================
llm_sb = client.messages.create(
    model=LLM_MODEL,
    max_tokens=LLM_MAX_TOKENS,
    thinking=LLM_THINKING,
    system=SUPABASE_DIGEST_SYS,
    messages=[{"role": "user", "content": sb_input}],
)

# === Gemini -- uncomment to use (and comment out the Anthropic block above) ===
# llm_sb = gemini_client.models.generate_content(
#     model=GEMINI_MODEL,
#     contents=sb_input,
#     config={"system_instruction": SUPABASE_DIGEST_SYS, "max_output_tokens": LLM_MAX_TOKENS},
# )

# === OpenAI -- uncomment to use ==============================================
# llm_sb = openai_client.chat.completions.create(
#     model=OPENAI_MODEL,
#     max_tokens=LLM_MAX_TOKENS,
#     messages=[{"role": "system", "content": SUPABASE_DIGEST_SYS},
#               {"role": "user",   "content": sb_input}],
# )

# === DeepSeek -- uncomment to use ============================================
# llm_sb = deepseek_client.chat.completions.create(
#     model=DEEPSEEK_MODEL,
#     max_tokens=LLM_MAX_TOKENS,
#     messages=[{"role": "system", "content": SUPABASE_DIGEST_SYS},
#               {"role": "user",   "content": sb_input}],
# )

raw_sb, thinking_sb = _extract(llm_sb)
m = re.search(r"\{.*\}", raw_sb, re.DOTALL)
supabase_digest = json.loads(m.group(0)) if m else None

print(f"Claude summarized the stats ({len(thinking_sb)} chars of thinking).")
print("Per-team stats digest for Step 5:\n")
print(json.dumps(supabase_digest, indent=2))


## Step 5 · Predict the result (the agent's own opinion)

Now the agent forms its **own** view of the match, combining the Sportmonks and Supabase digests into a single prediction: the most likely outcome, a probability, and a rationale.

**The market is deliberately left out here.** We want an opinion formed independently of Polymarket — otherwise the agent would just parrot the market price. Comparing this independent view against the market in Step 6 is exactly where any **edge** comes from.

In [ ]:
PREDICT_SYS = (
    "You are a soccer match analyst. You receive two pre-distilled digests for "
    "one fixture and must produce the agent's own outcome prediction.\n\n"

    "## Input shape\n"
    "  - fixture           : match name\n"
    "  - home_code         : home team short code (use as a JSON key for the home outcome)\n"
    "  - away_code         : away team short code (use as a JSON key for the away outcome)\n"
    "  - sportmonks_digest : digest of Sportmonks pre-match data (model probs, bookmaker\n"
    "                        consensus, expected goals). Values may be null when staging\n"
    "                        hasn't seeded the data — see its `data_availability` flag.\n"
    "  - supabase_digest   : digest of long-horizon priors (playing style, set-piece\n"
    "                        efficiency, group/KO goals-per-game). Note its sample-size\n"
    "                        caveats.\n\n"

    "## Output schema (return ONLY this JSON — no prose, no code fences)\n"
    "{\n"
    "  'fixture'    : str,\n"
    "  'outcome'    : str,                          // home_code | 'draw' | away_code — the most likely outcome\n"
    "  'probability': float,                        // 0..1; confidence in `outcome`\n"
    "  'rationale'  : str,                          // 1-3 sentences self-contained. Name the teams,\n"
    "                                               // the signals you leaned on, and major caveats.\n"
    "  'used_signals': {                            // for traceability into §6\n"
    "    'sportmonks' : 'leaned_on' | 'unavailable',\n"
    "    'supabase'   : 'leaned_on' | 'unavailable'\n"
    "  },\n"
    "  'confidence_level': 'high' | 'medium' | 'low'   // honest about how thin your evidence was\n"
    "}\n\n"

    "Be honest about uncertainty: if both digests are sparse, low confidence is the right answer. "
    "Do NOT consult the market (you don't have it). Probability must reflect what the priors say "
    "alone — anchoring to a market mid would defeat the point."
)

predict_input = json.dumps({
    "fixture":           fixture["name"],
    "home_code":         home["short_code"],
    "away_code":         away["short_code"],
    "sportmonks_digest": sportmonks_digest,
    "supabase_digest":   supabase_digest,
})

# === Anthropic (default) =====================================================
llm_predict = client.messages.create(
    model=LLM_MODEL,
    max_tokens=LLM_MAX_TOKENS,
    thinking=LLM_THINKING,
    system=PREDICT_SYS,
    messages=[{"role": "user", "content": predict_input}],
)

# === Gemini -- uncomment to use (and comment out the Anthropic block above) ===
# llm_predict = gemini_client.models.generate_content(
#     model=GEMINI_MODEL,
#     contents=predict_input,
#     config={"system_instruction": PREDICT_SYS, "max_output_tokens": LLM_MAX_TOKENS},
# )

# === OpenAI -- uncomment to use ==============================================
# llm_predict = openai_client.chat.completions.create(
#     model=OPENAI_MODEL,
#     max_tokens=LLM_MAX_TOKENS,
#     messages=[{"role": "system", "content": PREDICT_SYS},
#               {"role": "user",   "content": predict_input}],
# )

# === DeepSeek -- uncomment to use ============================================
# llm_predict = deepseek_client.chat.completions.create(
#     model=DEEPSEEK_MODEL,
#     max_tokens=LLM_MAX_TOKENS,
#     messages=[{"role": "system", "content": PREDICT_SYS},
#               {"role": "user",   "content": predict_input}],
# )

raw_pred, thinking_pred = _extract(llm_predict)
m = re.search(r"\{.*\}", raw_pred, re.DOTALL)
prediction = json.loads(m.group(0)) if m else None

print(f"The agent formed its own prediction ({len(thinking_pred)} chars of thinking):\n")
print(json.dumps(prediction, indent=2))
if prediction:
    print(f"\n-> In plain words: most likely '{prediction['outcome']}' at "
          f"{prediction['probability']:.0%} confidence ({prediction['confidence_level']}).")


## Step 6 · Decide whether to bet (turn the prediction into a trade)

Here the agent acts like a disciplined **bankroll manager** for a $100 play-money account. It compares its own prediction (Step 5) to the market (Step 3) and outputs a concrete decision.

The key idea is **edge** = (the agent's probability) − (the market's implied probability) for the same outcome:

- **Positive edge** → the market is *underpricing* the pick → consider going **long** (back it).
- **Negative edge** → the market is *overpricing* it → consider going **short** (fade it).
- **Tiny edge** (noise) → don't trade.

Position size scales with the edge and the agent's confidence (and shrinks when confidence is low). With a small wallet and weak conviction, **not betting is a perfectly good answer.**

In [ ]:
STRATEGY_SYS = (
    "You are a bankroll manager for a $100 demo account. You receive the agent's "
    "own prediction and the current Polymarket market view, and decide whether "
    "to trade and on what terms.\n\n"

    "## Input shape\n"
    "  - prediction        : {outcome, probability, confidence_level, rationale, …}\n"
    "                        The agent's own view, formed without seeing the market.\n"
    "  - polymarket_digest : {implied_win_prob, sum_implied_prob, execution_handles,\n"
    "                        market_handle, data_availability, summary}.\n"
    "                        The market's view (implied_win_prob keys match team codes).\n\n"

    "## How to decide\n"
    "  1. Edge = prediction.probability − polymarket_digest.implied_win_prob[prediction.outcome]\n"
    "     (in percentage points). Positive edge = market UNDER-prices the pick → long.\n"
    "     Negative edge = market OVER-prices the pick → short.\n"
    "  2. Size discipline (max $5 per trade, $100 wallet):\n"
    "       |edge| < 5pp                  → don't trade (noise)\n"
    "       |edge| 5-15pp                 → $1-2  (modest position)\n"
    "       |edge| > 15pp                 → $3-5  (high-conviction position)\n"
    "     Then HALVE the size if confidence_level is 'low'.\n"
    "       confidence 'medium'           → use the size above\n"
    "       confidence 'high'             → use up to 1.5× (capped at $5)\n"
    "     If the Polymarket digest's data_availability is not 'mids_available', skip — you\n"
    "     can't price an edge without mids.\n"
    "  3. limit_price is the worst price per share you'll accept. For long, a bit ABOVE the\n"
    "     current mid for the YES token (e.g. mid 0.665 → limit 0.68). For short, a bit\n"
    "     ABOVE the current mid for the NO token, which is (1 − mid_yes) (e.g. mid_yes 0.665\n"
    "     → NO_mid 0.335 → limit 0.36).\n\n"

    "## Output schema (return ONLY this JSON — no prose, no code fences)\n"
    "{\n"
    "  'should_trade'   : bool,\n"
    "  'outcome'        : str,                    // echo prediction.outcome — what to trade\n"
    "  'direction'      : 'long' | 'short',       // long = back the outcome; short = fade it\n"
    "  'size_usdc'      : float,                  // 0 when not trading; ≤5 for this demo\n"
    "  'limit_price'    : float,                  // 0..1; see rule 3 above\n"
    "  'edge_pp'        : float,                  // (agent_prob − market_prob) × 100\n"
    "  'market_handle'  : str,                    // echo polymarket_digest.market_handle for traceability\n"
    "  'rationale'      : str                     // 1-3 sentences self-contained: state the edge,\n"
    "                                             // the size logic, the limit_price logic.\n"
    "}\n\n"

    "Be conservative: small wallet, weak conviction → skipping is a valid answer."
)

strategy_input = json.dumps({
    "prediction":        prediction,
    "polymarket_digest": polymarket_digest,
})

# === Anthropic (default) =====================================================
llm_strategy = client.messages.create(
    model=LLM_MODEL,
    max_tokens=LLM_MAX_TOKENS,
    thinking=LLM_THINKING,
    system=STRATEGY_SYS,
    messages=[{"role": "user", "content": strategy_input}],
)

# === Gemini -- uncomment to use (and comment out the Anthropic block above) ===
# llm_strategy = gemini_client.models.generate_content(
#     model=GEMINI_MODEL,
#     contents=strategy_input,
#     config={"system_instruction": STRATEGY_SYS, "max_output_tokens": LLM_MAX_TOKENS},
# )

# === OpenAI -- uncomment to use ==============================================
# llm_strategy = openai_client.chat.completions.create(
#     model=OPENAI_MODEL,
#     max_tokens=LLM_MAX_TOKENS,
#     messages=[{"role": "system", "content": STRATEGY_SYS},
#               {"role": "user",   "content": strategy_input}],
# )

# === DeepSeek -- uncomment to use ============================================
# llm_strategy = deepseek_client.chat.completions.create(
#     model=DEEPSEEK_MODEL,
#     max_tokens=LLM_MAX_TOKENS,
#     messages=[{"role": "system", "content": STRATEGY_SYS},
#               {"role": "user",   "content": strategy_input}],
# )

raw_strat, thinking_strat = _extract(llm_strategy)
m = re.search(r"\{.*\}", raw_strat, re.DOTALL)
strategy = json.loads(m.group(0)) if m else None

print(f"The agent decided on a strategy ({len(thinking_strat)} chars of thinking):\n")
print(json.dumps(strategy, indent=2))
if strategy and strategy.get("should_trade"):
    print(f"\n-> In plain words: {strategy['direction'].upper()} ${strategy['size_usdc']:.2f} on "
          f"'{strategy['outcome']}' (edge {strategy['edge_pp']:+.1f} points).")
elif strategy:
    print(f"\n-> In plain words: no trade -- edge {strategy.get('edge_pp', 0):+.1f} points "
          f"isn't worth it for this wallet.")


## Step 7 · Place the bet (open a position)

If Step 6 decided to trade, we build an order and POST it to the arena. If it decided to skip, we do nothing — **predict-only runs are fully supported**, and the reasoning still gets recorded in Step 8.

Either way we print the order payload so you can see its shape. (The arena `/orders` endpoint may not be live on staging yet, so the POST can 404 — that's expected here.)

An **idempotency key** (a random UUID) is included so that retrying the same request can't accidentally place the order twice.

In [ ]:
if strategy and strategy.get("should_trade"):
    order_payload = {
        "fixture_code":           str(SPORTMONKS_FIXTURE_ID),     # canonical fixture handle (TBD whether arena keys by sportmonks_id or its own)
        "outcome":                strategy["outcome"],
        "direction":              strategy["direction"],
        "size_usdc":              f"{strategy['size_usdc']:.2f}",
        "limit_price":            strategy["limit_price"],
        "time_in_force_seconds":  30,
        "idempotency_key":        str(uuid.uuid4()),
    }
    print("Strategy says TRADE. Here's the exact order we'd submit:\n")
    print(json.dumps(order_payload, indent=2))
    try:
        r = requests.post(
            f"{ARENA}/api/v1/orders",       # path TBD — adjust when arena ships /orders
            headers=H_ARENA, timeout=10,
            json=order_payload,
        )
        if r.status_code == 404:
            print("\nHTTP 404 -- the arena /orders endpoint isn't live on staging yet. "
                  "Expected for now; the payload above is what a real run would send.")
        elif r.ok:
            print(f"\nHTTP {r.status_code} (OK) -- order accepted.")
        else:
            print(f"\nHTTP {r.status_code} -- order rejected. Body: {r.text[:300]}")
        order_response = r.json() if r.ok else None
    except Exception as e:
        print(f"\nOrder POST failed: {type(e).__name__}: {e}")
        order_response = None
else:
    print("Strategy says DON'T trade, so we skip placing an order.")
    print("Predict-only runs are fully supported -- Step 8 still records everything.")
    order_response = None


## Step 8 · Record the agent's reasoning (the ledger)

Finally, the agent writes a **ledger**: a structured, step-by-step record of everything it just did. The arena reads this to audit, verify, and score your agent — so a well-formed ledger is how you actually "submit" your work.

Each record is one node in a graph (`upstream_record_id` links a step to the steps it depended on). The behavior types:

| Behavior | Meaning |
|----------|---------|
| `Observing` | What triggered the run (here, a pretend cron trigger) |
| `ToolCalling` | An external data call (Sportmonks, Polymarket, Supabase) |
| `Thinking` | An LLM step (each digest, the prediction, the strategy) |
| `Acting` | A committed decision — the prediction (always) and an order (only if we traded) |

This run produces **14 records** (15 when an order is placed). We build them with plain dicts (no SDK) and POST them as one batch. `agent_id` isn't set here — the arena fills it in server-side from your `ARENA_KEY`. As with Step 7, the endpoint may 404 on staging; the script reports rather than crashes.

In [ ]:
LEDGER_SESSION_ID = f"prematch:{SPORTMONKS_FIXTURE_ID}"

def _new_record(behavior, **fields):
    """Compose the BaseRecord envelope + behavior-specific fields.

    Note: agent_id is intentionally omitted. The arena resolves it server-side
    from the x-api-key on POST, so wire records do not carry it. The local
    dump produced by this script mirrors that — schema-wise, agent_id is
    required, but it only becomes present after the server enriches the
    record."""
    rec = {
        "schema_version": LEDGER_SCHEMA_VERSION,
        "session_id":     LEDGER_SESSION_ID,
        "record_id":      str(uuid.uuid4()),
        "behavior":       behavior,
        "client_ts_utc":  int(time.time() * 1000),
    }
    rec.update({k: v for k, v in fields.items() if v is not None})
    return rec

def _mi(resp):
    """Build a ModelInvocation dict from ANY provider's response (Anthropic /
    OpenAI / DeepSeek / Gemini). Captures the `internal_reasoning` trace when the
    provider exposes one -- see schema v0.3 ModelInvocation."""
    _, thinking = _extract(resp)
    if hasattr(resp, "usage") and hasattr(resp.usage, "input_tokens"):          # Anthropic
        provider, model = "anthropic", LLM_MODEL
        tokens_in, tokens_out = resp.usage.input_tokens, resp.usage.output_tokens
    elif hasattr(resp, "usage") and hasattr(resp.usage, "prompt_tokens"):       # OpenAI / DeepSeek
        model = getattr(resp, "model", "") or ""
        provider = "deepseek" if "deepseek" in model else "openai"
        tokens_in, tokens_out = resp.usage.prompt_tokens, resp.usage.completion_tokens
    elif hasattr(resp, "usage_metadata"):                                       # Gemini
        provider, model = "gemini", GEMINI_MODEL
        um = resp.usage_metadata
        tokens_in  = getattr(um, "prompt_token_count", None)
        tokens_out = getattr(um, "candidates_token_count", None)
    else:
        provider, model, tokens_in, tokens_out = "unknown", "", None, None
    mi = {"provider": provider, "model_name": model,
          "tokens_in": tokens_in, "tokens_out": tokens_out}
    if thinking:
        mi["internal_reasoning"] = thinking
    return mi

def _trunc(obj, limit=30000):
    """JSON-stringify + truncate to keep individual fields under SDK size limits
    (Thinking.output_payload ≤ 32 KB; per-record JSON ≤ 64 KB)."""
    s = obj if isinstance(obj, str) else json.dumps(obj, default=str)
    return s if len(s) <= limit else s[:limit] + f"…[truncated, was {len(s)} chars]"


# (1) Observing — synthetic cron trigger that woke the agent.
rec_trigger = _new_record(
    "Observing",
    trigger_source="dev-guide-workflow-test",
    trigger_type="cron_trigger",
    trigger_description=f"Pre-match prediction run for fixture {SPORTMONKS_FIXTURE_ID} ({fixture['name']})",
    trigger_payload_summary=(
        f"fixture_id={SPORTMONKS_FIXTURE_ID}; window=PRE_MATCH; "
        f"kickoff_utc={fixture['starting_at']}; home={home['short_code']}; away={away['short_code']}"
    ),
)

# (2) ToolCalling — Sportmonks schedule
rec_sm_schedule = _new_record(
    "ToolCalling",
    upstream_record_id=[rec_trigger["record_id"]],
    tool_meta={"name": "sportmonks", "endpoint": "/v3/football/schedules/seasons/{season_id}",
               "via": "arena.sportmonks_proxy"},
    description="List WC2026 season schedule to discover fixtures",
    input_payload={"season_id": 26618},
    output_payload={"stage_count": len(schedule), "picked_fixture_id": SPORTMONKS_FIXTURE_ID},
    success=True,
)

# (3) ToolCalling — Sportmonks fixture detail
rec_sm_fixture = _new_record(
    "ToolCalling",
    upstream_record_id=[rec_sm_schedule["record_id"]],
    tool_meta={"name": "sportmonks", "endpoint": "/v3/football/fixtures/{fixture_id}",
               "via": "arena.sportmonks_proxy"},
    description="Fetch fixture detail with pre-match prediction includes",
    input_payload={"fixture_id": SPORTMONKS_FIXTURE_ID,
                   "include":    "participants;predictions;odds;xGFixture"},
    output_payload={
        "fixture_name":      fixture["name"],
        "kickoff_utc":       fixture["starting_at"],
        "participants":      [{"id": p["id"], "name": p["name"],
                               "short_code": p["short_code"],
                               "country_id": p["country_id"],
                               "location": p["meta"]["location"]} for p in fixture["participants"]],
        "predictions_count": len(fixture.get("predictions") or []),
        "odds_count":        len(fixture.get("odds") or []),
        "xgfixture_count":   len(fixture.get("xgfixture") or []),
    },
    success=True,
)

# (4) Thinking — Sportmonks digest
rec_th_sportmonks = _new_record(
    "Thinking",
    upstream_record_id=[rec_sm_fixture["record_id"]],
    model_invocation=_mi(llm_digest),
    prompt=_trunc(DIGEST_SYS, limit=16000),
    inputs=[{
        "input_record_id": rec_sm_fixture["record_id"],
        "input_payload":   _trunc({
            "fixture":     fixture["name"],
            "home_code":   home["short_code"],
            "away_code":   away["short_code"],
            "predictions": fixture.get("predictions"),
            "odds":        fixture.get("odds"),
            "xGFixture":   fixture.get("xgfixture"),
        }),
    }],
    output_payload=_trunc(sportmonks_digest),
)

# (5a) ToolCalling — arena: look up the polymarket event slug for the fixture.
rec_pm_slug = _new_record(
    "ToolCalling",
    upstream_record_id=[rec_sm_schedule["record_id"]],
    tool_meta={"name": "arena-mapping",
               "endpoint": "/api/v1/web/mapping"},
    description="Look up curated Polymarket event_slug for this Sportmonks fixture",
    input_payload={"fixture_id": SPORTMONKS_FIXTURE_ID},
    output_payload={"polymarket_event_slug": polymarket_event_slug},
    success=polymarket_event_slug is not None,
)

# (5b) ToolCalling — Polymarket Gamma: fetch the event + nested markets
# (condition_ids + clobTokenIds for home / draw / away).
rec_pm_event = _new_record(
    "ToolCalling",
    upstream_record_id=[rec_pm_slug["record_id"]],
    tool_meta={"name": "polymarket-gamma",
               "endpoint": "/api/v1/data/proxy/polymarket-gamma/events",
               "via": "arena.proxy"},
    description="Fetch Polymarket event + 3 child winner markets by slug",
    input_payload={"slug": polymarket_event_slug},
    output_payload={
        "outcomes": {k: {"team_code":     moneyline["outcomes"][k]["team_code"],
                         "condition_id":  moneyline["outcomes"][k]["condition_id"],
                         "token_yes":     moneyline["outcomes"][k]["token_yes"]}
                     for k in moneyline["outcomes"]}
    } if moneyline else None,
    success=moneyline is not None,
)

# (5c) ToolCalling — Polymarket CLOB: live midpoint per YES token (3 calls
# summarized into one record).
rec_pm_mids = _new_record(
    "ToolCalling",
    upstream_record_id=[rec_pm_event["record_id"]],
    tool_meta={"name": "polymarket-clob",
               "endpoint": "/api/v1/data/proxy/polymarket-clob/midpoint",
               "via": "arena.proxy"},
    description="Fetch CLOB midpoint per outcome YES token (home / draw / away)",
    input_payload={"token_ids": [
        moneyline["outcomes"][k]["token_yes"] for k in moneyline["outcomes"]
    ] if moneyline else None},
    output_payload={
        k: moneyline["outcomes"][k]["current_mid_yes"] for k in moneyline["outcomes"]
    } if moneyline else None,
    success=moneyline is not None,
)

# (6) Thinking — Polymarket digest
rec_th_polymarket = _new_record(
    "Thinking",
    upstream_record_id=[rec_pm_slug["record_id"],
                        rec_pm_event["record_id"],
                        rec_pm_mids["record_id"]],
    model_invocation=_mi(llm_pm),
    prompt=_trunc(POLYMARKET_DIGEST_SYS, limit=16000),
    inputs=[{
        "input_record_id": rec_pm_mids["record_id"],
        "input_payload":   _trunc(moneyline),
    }],
    output_payload=_trunc(polymarket_digest),
)

# (7) ToolCalling — Supabase catalog discovery
rec_sb_catalog = _new_record(
    "ToolCalling",
    upstream_record_id=[rec_trigger["record_id"]],
    tool_meta={"name": "supabase", "endpoint": "/rest/v1/catalog_full"},
    description="Discover available Supabase tables via the public catalog",
    input_payload={"params": {"select": "table_name,category,row_count,table_description",
                              "order":  "category,table_name"}},
    output_payload={"available_tables": [t["table_name"] for t in catalog],
                    "count": len(catalog)},
    success=True,
)

# (8) ToolCalling — Supabase priors fetch
rec_sb_priors = _new_record(
    "ToolCalling",
    upstream_record_id=[rec_sb_catalog["record_id"], rec_sm_fixture["record_id"]],
    tool_meta={"name": "supabase", "endpoint": f"/rest/v1/{WANTED_TABLE}",
               "schema": "world_cup_arena"},
    description=f"Fetch {WANTED_TABLE} priors for both teams",
    input_payload={"country_id": f"in.({TEAM_A_ID},{TEAM_B_ID})", "select": "*"},
    output_payload=priors_rows,
    success=True,
)

# (9) Thinking — Supabase digest
rec_th_supabase = _new_record(
    "Thinking",
    upstream_record_id=[rec_sb_priors["record_id"]],
    model_invocation=_mi(llm_sb),
    prompt=_trunc(SUPABASE_DIGEST_SYS, limit=16000),
    inputs=[{
        "input_record_id": rec_sb_priors["record_id"],
        "input_payload":   _trunc({
            "fixture":      fixture["name"],
            "source_table": WANTED_TABLE,
            "home_code":    home["short_code"],
            "away_code":    away["short_code"],
            "rows":         priors_rows,
        }),
    }],
    output_payload=_trunc(supabase_digest),
)

# (10) Thinking — Predict (priors only, blind to market).
# The reasoning lives here; the structured prediction is committed via the
# Acting record below, which is the form the arena validates + scores.
rec_th_predict = _new_record(
    "Thinking",
    upstream_record_id=[rec_th_sportmonks["record_id"], rec_th_supabase["record_id"]],
    model_invocation=_mi(llm_predict),
    prompt=_trunc(PREDICT_SYS, limit=16000),
    inputs=[
        {"input_record_id": rec_th_sportmonks["record_id"],
         "input_payload":   _trunc(sportmonks_digest)},
        {"input_record_id": rec_th_supabase["record_id"],
         "input_payload":   _trunc(supabase_digest)},
    ],
    output_payload=_trunc(prediction),
)

# (11) Acting — Prediction (validated + scored by the arena).
# Per the new ledger contract, predictions are emitted as Acting records with
# action_type="prediction" and structured `parameters` the arena snapshots
# for scoring at settlement. probability is clamped to the schema range
# [0.001, 0.999].
_pred_prob = max(0.001, min(0.999, float(prediction["probability"])))
rec_act_predict = _new_record(
    "Acting",
    upstream_record_id=[rec_th_predict["record_id"]],
    action_type=     "prediction",
    target_system=   "arena",
    action_summary=  f"Predict {prediction['outcome']} @ p={_pred_prob:.2f} for fixture {SPORTMONKS_FIXTURE_ID}",
    parameters=      {
        "fixture_code": str(SPORTMONKS_FIXTURE_ID),
        "outcome":      prediction["outcome"],
        "probability":  _pred_prob,
    },
    dry_run=         False,
    execution_status="confirmed",
)

# (12) Thinking — Strategy (prediction + market → trade decision)
rec_th_strategy = _new_record(
    "Thinking",
    upstream_record_id=[rec_th_predict["record_id"], rec_th_polymarket["record_id"]],
    model_invocation=_mi(llm_strategy),
    prompt=_trunc(STRATEGY_SYS, limit=16000),
    inputs=[
        {"input_record_id": rec_th_predict["record_id"],
         "input_payload":   _trunc(prediction)},
        {"input_record_id": rec_th_polymarket["record_id"],
         "input_payload":   _trunc(polymarket_digest)},
    ],
    output_payload=_trunc(strategy),
)

records = [
    rec_trigger, rec_sm_schedule,
    rec_pm_slug, rec_pm_event, rec_pm_mids,
    rec_sm_fixture, rec_th_sportmonks,
    rec_th_polymarket,
    rec_sb_catalog, rec_sb_priors, rec_th_supabase,
    rec_th_predict, rec_act_predict, rec_th_strategy,
]

# (13) Acting — emit only when the agent actually submitted an order.
# This is the AGENT-side Acting (intent / submission). The arena will
# additionally write its own Acting record(s) server-side at fill / close
# time with target_system="public-chain" + execution_id=<tx_hash>. The two
# are independent evidence of the same logical action.
if strategy and strategy.get("should_trade"):
    # Did the order POST land cleanly? If yes, status=pending (waiting on fill);
    # if not, status=failed. order_response is None on 404 / exception.
    submitted_ok = isinstance(order_response, dict) and bool(order_response)
    rec_act = _new_record(
        "Acting",
        upstream_record_id=[rec_th_strategy["record_id"]],
        action_type=     "open_order",
        target_system=   "arena",     # we submit to arena; arena routes to polymarket-clob
        action_summary=  (f"Open {strategy['direction']} ${strategy['size_usdc']:.2f} on "
                          f"{strategy['outcome']} @ ≤{strategy['limit_price']}"),
        parameters=      order_payload,
        dry_run=         False,
        execution_status="pending" if submitted_ok else "failed",
        execution_id=    (order_response.get("order_id") if submitted_ok else None),
    )
    records.append(rec_act)

print(f"Built {len(records)} ledger records -- one per step the agent took:\n")
for rec in records:
    label = (rec.get("description")
             or rec.get("action_summary")
             or rec.get("trigger_description")
             or rec.get("prompt", "")[:50])
    print(f"  {rec['behavior']:12s} {rec['record_id'][:8]}...  {label}")

# Submit the trace as a single batch. Per the new ledger contract:
#   - No session-create endpoint; session_id is purely a client-side string.
#   - Bare record dicts (no {"body": {...}} envelope).
#   - agent_id is derived server-side from x-api-key.
#   - One round-trip per cycle via /records/batch (≤50 records). Response:
#       {"records": [<enriched echoes>], "errors": [{index, code, message}, ...]}
# Endpoint isn't live on staging yet — expect 404. Script reports rather than raises.
try:
    r = requests.post(
        f"{ARENA}/api/v1/arena/ledger/records/batch",
        headers=H_ARENA, timeout=15,
        json={"records": records},
    )
    if r.status_code == 404:
        print(f"\nHTTP 404 -- the ledger endpoint isn't live on staging yet (expected). "
              f"The {len(records)} records above are exactly what a real run would submit.")
    elif r.ok:
        resp = r.json()
        print(f"\nHTTP {r.status_code} (OK) -- ledger accepted: "
              f"{len(resp.get('records', []))} stored, {len(resp.get('errors', []))} error(s).")
        for e in resp.get("errors", []):
            print(f"    [#{e.get('index')}] {e.get('code')}: {e.get('message')}")
    else:
        print(f"\nHTTP {r.status_code} -- ledger rejected. Body: {r.text[:300]}")
except Exception as e:
    print(f"\nLedger POST failed: {type(e).__name__}: {e}")
